# <b><div style='padding:15px;background-color:#850E35;color:white;border-radius:2px;font-size:110%;text-align: center'>Transformers for TensorFlow</div></b>
![](https://1.bp.blogspot.com/-qQryqABhdhA/XcC3lJupTKI/AAAAAAAAAzA/MOYu3P_DFRsmNkpjD9j813_SOugPgoBLACLcBGAsYHQ/s1600/h1.png)

This notebook walks you through how to work with Transformers using TensorFlow.

# <b><div style='padding:15px;background-color:#850E35;color:white;border-radius:2px;font-size:110%;text-align: center'>1. Loading Dataset</div></b>

First, let's install the `datasets` library. 

In [1]:
# Installing the "datasets" library
!pip install -q datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 41.2 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
pylibcudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
cudf-cu12 25.2.2 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 22.0.0 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
cudf-polars-cu12 25.6.0 requires pylibcudf-cu12==25.6.*, but you have pylibcudf-cu12 25.2.2 which is incompatible.


And then let's use this library to load the `rotten_tomatoes` dataset, which presumably contains movie reviews or related data. 

In [2]:
# Importing the necessary function to load a dataset
from datasets import load_dataset

# Loading the "rotten_tomatoes" dataset
dataset = load_dataset("rotten_tomatoes")

README.md: 0.00B [00:00, ?B/s]

train.parquet:   0%|          | 0.00/699k [00:00<?, ?B/s]

validation.parquet:   0%|          | 0.00/90.0k [00:00<?, ?B/s]

test.parquet:   0%|          | 0.00/92.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/8530 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1066 [00:00<?, ? examples/s]

Let's explore this dataset: 

In [3]:
# Displaying the loaded dataset
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

Let's take a look at the first example from the "test" split of the dataset.

In [4]:
# Accessing the first example from the test split of the dataset
dataset["test"][0]

{'text': 'lovingly photographed in the manner of a golden book sprung to life , stuart little 2 manages sweetness largely without stickiness .',
 'label': 1}

# <b><div style='padding:15px;background-color:#850E35;color:white;border-radius:2px;font-size:110%;text-align: center'>2. Data Preprocessing</div></b>

Let's initialize for the `distilbert-base-uncased` model. 

In [5]:
# Importing the tokenizer for a pre-trained model
from transformers import AutoTokenizer

# Initializing the tokenizer for the "distilbert-base-uncased" model
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

For example, let's tokenize the text of the first training example in the dataset using this tokenizer.

In [6]:
# Tokenizing the text of the first training example
tokenizer(dataset["train"][0]["text"])

{'input_ids': [101, 1996, 2600, 2003, 16036, 2000, 2022, 1996, 7398, 2301, 1005, 1055, 2047, 1000, 16608, 1000, 1998, 2008, 2002, 1005, 1055, 2183, 2000, 2191, 1037, 17624, 2130, 3618, 2084, 7779, 29058, 8625, 13327, 1010, 3744, 1011, 18856, 19513, 3158, 5477, 4168, 2030, 7112, 16562, 2140, 1012, 102], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

Let's define a preprocessing function named preprocess_function. This function takes a dictionary of examples as input and tokenizes the "text" field with truncation. 

In [7]:
# Preprocessing function for tokenization
def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True)

After that let's use the map function to apply this preprocessing function to the entire dataset in batches.

In [8]:
# Applying the preprocessing function to the entire dataset in batches
dataset = dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/8530 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Let's create a batch of examples from the dataset.

In [9]:
# Importing the data collator with padding
from transformers import DataCollatorWithPadding

# Initializing the data collator with padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="tf")

2026-01-01 16:30:29.880258: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767285030.054875      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767285030.106248      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

# <b><div style='padding:15px;background-color:#850E35;color:white;border-radius:2px;font-size:110%;text-align: center'>3. Model Loading</div></b>

Let's load the `distilbert` model for our sentiment analysis.

In [10]:
# Importing the TensorFlow version of the model for sequence classification
from transformers import TFAutoModelForSequenceClassification

# Initializing a model for sequence classification using "distilbert-base-uncased"
my_model = TFAutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

I0000 00:00:1767285049.526134      47 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15513 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0
TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_projector.bias', 'vocab_layer_norm.weight', 'vocab_layer_norm.bias', 'vocab_transform.bias', 'vocab_transform.weight']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model

Let's prepare training and validation datasets as TensorFlow datasets. 

In [11]:
# Preparing the training dataset as a TensorFlow dataset
tf_train_set = my_model.prepare_tf_dataset(
    dataset["train"],
    shuffle=True,
    batch_size=16,
    collate_fn=data_collator,
)

# Preparing the validation dataset as a TensorFlow dataset
tf_validation_set = my_model.prepare_tf_dataset(
    dataset["validation"],
    shuffle=False,
    batch_size=16,
    collate_fn=data_collator,
)

# <b><div style='padding:15px;background-color:#850E35;color:white;border-radius:2px;font-size:110%;text-align: center'>4. Model Training</div></b>

Let's compile the model using the Adam optimizer with a learning rate of 3e-5.

In [13]:
# Importing the Adam optimizer from TensorFlow
# from tensorflow.keras.optimizers import Adam
from tf_keras.optimizers import Adam # tf_keras'ın yeni sürümünü kullanalım.

# Compiling the model with the Adam optimizer and a specified learning rate
my_model.compile(optimizer=Adam(3e-5))  # No loss argument!

Let's train the model using the prepared training and validation datasets for 2 epochs.

In [14]:
# Training the model
my_model.fit(x=tf_train_set, validation_data=tf_validation_set, epochs=2)

Epoch 1/2


I0000 00:00:1767285212.039758     157 service.cc:148] XLA service 0x7affb0dc6170 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1767285212.040061     157 service.cc:156]   StreamExecutor device (0): Tesla P100-PCIE-16GB, Compute Capability 6.0
I0000 00:00:1767285212.106835     157 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1767285212.240155     157 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


533/533 [==============================] - 69s 87ms/step - loss: 0.4188 - val_loss: 0.3410
Epoch 2/2
533/533 [==============================] - 37s 69ms/step - loss: 0.2143 - val_loss: 0.3629


# <b><div style='padding:15px;background-color:#850E35;color:white;border-radius:2px;font-size:110%;text-align: center'>5. Prediction</div></b>

Let's get a sample text for inference.

In [16]:
# Defining a text for inference
text = "I love NLP. It's fun to analyze the NLP tasks with Hugging Face."

Let's preprocess our text for passing our model.

In [17]:
# Tokenizing the text for inference
tokenized_text = tokenizer(text, return_tensors="tf")
tokenized_text 

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.


{'input_ids': <tf.Tensor: shape=(1, 21), dtype=int32, numpy=
array([[  101,  1045,  2293, 17953,  2361,  1012,  2009,  1005,  1055,
         4569,  2000, 17908,  1996, 17953,  2361,  8518,  2007, 17662,
         2227,  1012,   102]], dtype=int32)>, 'attention_mask': <tf.Tensor: shape=(1, 21), dtype=int32, numpy=
array([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]],
      dtype=int32)>}

Next, let's compute the model's logits (raw output scores) for the tokenized text.

In [18]:
# Obtaining model logits for the tokenized text
logits = my_model(**tokenized_text).logits

Lastly, let's print the index of the class with the highest logit score.

In [19]:
# Importing the math module from TensorFlow
from tensorflow import math

# Finding the index of the class with the highest logit score
int(math.argmax(logits, axis=-1)[0])

1

Thanks for reading. If you enjoyed this notebook, don't forget to upvote ☺️

Let's connect [YouTube](http://youtube.com/tirendazacademy) | [Medium](http://tirendazacademy.medium.com) | [Twitter](http://twitter.com/tirendazacademy) | [Linkedin](https://www.linkedin.com/in/tirendaz-academy) 😎